## Source code to test the quantum aspect of the application (Chapter 2)

**Chapter1** is goint to be the introduction on what quantum computing is and some basic notion that we need to discuss in advanced: 
- *Quantum Principles* 
- *Quantum operations*
- *Bell states and entanglement*
- *Core Theorem for the Teleportation protocol* (non copying theorem, and the entaglement spoky action at a distance)


#### Comparing the QKD to the TELEPORTATION protocol:
- Loading the pdf and the text sample
- Using as Cryptographyc algorithm a code block basic algorithm like AES
- Sending the key using a BB84 (basic of QKD accepted)
- Sending the key using teleportation protocol 
- Comparing the differences between them (number of bits needed, latency, security) -> plot of measures that can used 
- Plotting measures of the cryptographyc metrics to see the impact of the key transition from one source to another. 

### Additional information about the BB84 for the writting part 
| Resource Name | link | 
| ------- | -------- |
BB84 | https://arxiv.org/html/2312.05609v1
BB84 in practice | https://www.mdpi.com/2624-960X/7/3/35 
QKD concept | https://resources.geant.org/wp-content/uploads/2024/02/GN5-1_White-Paper_QKD-Concepts-and-Considerations.pdf
Symmetric key enc | https://arxiv.org/pdf/1405.0398 
AES | https://arxiv.org/pdf/1501.01427
Symm vs Asymm | http://paper.ijcsns.org/07_book/201901/20190107.pdf
Telep | https://hal.science/hal-01833706v1/file/paper.pdf
 


In [3]:
### imports of the app
from pypdf import PdfReader
import os


In [11]:
# reading the text from the pdf
# @F_PATH -> path to the pdf file
def pdf_to_str(F_PATH: str) -> str:
    reader = PdfReader(F_PATH)
    print(len(reader.pages))
    return reader.pages[3].extract_text()

def txt_to_str(F_PATH:str) -> str:
    text = ''
    with open(F_PATH, 'r') as file:
        text = file.read()
    return text 
# CWD -> current working directory
# depth -> for how deep the source code find itself relative to the root of the project
depth = 1
CWD: str = '/'.join(os.getcwd().split('/')[:-depth]) 
TXT_DIR: str = 'text/in.txt'
PDF_ABSPATH: str = f'{CWD}/{TXT_DIR}'


## importing and use an already-made excryption standard algorithm 
txt_to_str(f'{CWD}/{TXT_DIR}')


'The revised text first appeared in Great Britain in a three-volume\nhardcover ‘Second Edition’ from Allen & Unwin on 27 October 1966.\nBut again there were problems. Although the revisions Tolkien sent\nto America of the text itself were available to be utilized in the new\nBritish edition, his extensive revisions to the appendices were lost\nafter being entered into the Ballantine edition. Allen & Unwin were\nforced to reset the appendices using the copy as published in the first\nBallantine edition. This did not include Tolkien’s second, small set\nof revisions sent to Ballantine; but, more significantly, it did include\na great number of errors and omissions, many of which were not\ndiscovered until long afterwards. Thus, in the appendices, a close\nscrutiny of the first edition text and of the much later corrected\nimpressions of the second edition is necessary to discern whether any\nparticular change in this edition is authorial or erroneous.\nIn America, the revised text appear

In [ ]:
### BB84 protocol for QKD 
from qiskit import QuantumCircuit, transpile

# the bits and bases are both lists
# bits _> [0, 1, 0, 1, ...]
# bases _> [true, false, ...] true for the 0/1 base and false for +/- base
def encode(bits: list[int], bases: list[bool]) -> list[int]:
    for bit, base in zip(bits, bases):
        circ = QuantumCircuit(1, 1)
        # for Z-base we apply an X Gate
        if base:
            if bit == 1:
                circ.x(0)
        # for the X-base we apply an additional H Gate on both qbits
        else:
            if bit == 0:
                circ.h(0)
            else:
                circ.x(0)
                circ.h(0)


def filter_bybases(bits: list[int], a_bases: list[int], b_bases:list[int]) -> list[int]:
    pass 




11


'TH  E  L  ORD  OF  THE  RI  NGS  \nSince 1974, Christopher Tolkien has sent additional corrections, \nas errors have been discovered, to the British publishers of The Lord \nof the Rings (Allen & Unwin, later Unwin Hyman, and now Harper-\nCollins), who have tried to be conscientious in the impossible task \nof maintaining a textual integrity in whichever editions of The Lord \nof the Rings they have published. However, every time the text has \nbeen reset for publication in a new format (e.g. the various paperback \neditions published in England in the 1970s and 1980s), huge numbers \nof new misprints have crept in, though at times some of these errors \nhave been observed and corrected in later printings. Still, throughout \nthese years the three-volume British hardcover edition has retained \nthe highest textual integrity. \nIn the United States, the text of the Ballantine paperback has \nremained unchanged for more than three decades after Tolkien added \nhis few revisions in 1966.

In [42]:
# QKD approach implementation 
# For a series of bits 0, 1 we take two measurement bases either the normal one |0>, |1> or the |+>, |-> 
# For the second bases we apply to the circuit a Hadamard gate to bring the qbit into the base
#!! FOR PRACTICAL PURPOSES I CAN CREATE A FULL CIRCUIT TO BE DRAWN TO DISPLAY HOW IT WILL LOOK
import numpy as np
import matplotlib.pyplot as plt 
import qiskit as qsk 
from qiskit_aer import AerSimulator
from qiskit import QuantumCircuit
import copy
from collections import deque

type Circs = list[QuantumCircuit]
type Base = int 
type Bases = list[int]
type Bits = list[Bit]
type Bit = 0 | 1

class Upper_Block:
    __i: int = 0
    def __init__(self, indeces: Bits, bit_sum: int, iter: int):
        self.id = f'BLK_{Upper_Block.__i}'
        Upper_Block.__i += 1
        self.indeces = indeces
        self.iter = iter 
        self.bit_sum = bit_sum
        self.parity = -1

    @classmethod
    def reset(cls):
        cls.__i = 0
    
    def __str__(self) -> str:
        return f'Block: {self.id} | Indeces: {self.indeces} | Iter: {self.iter} | Bit_Sum: {self.bit_sum} | Parity: {self.parity}'



def encode_persending(circuits: Circs, bases: Bases, bits: list[int]) -> None:
    if len(circuits) != 0:
        circuits.clear()

    if len(bases) != len(bits):
        raise Exception("####### Error in the encode_persending function <:> Length of the bases list should be equal to the length of bits list")

    for base, bit in zip(bases, bits):
        circ = QuantumCircuit(1, 1)
        # Z-base (vertical - |0> & |1>)
        if base == 0:
            # Apply a PauliX Gate on the qbit
            if bit == 1:
                circ.x(0)
        # X-base (horizontal - |+> & |->)
        else:
            if bit == 1:
                circ.x(0)
            circ.h(0)
        # Add the circuit to the list of circuits to be run after by the simulator
        circuits.append(circ)




# For the measurement we simple choose a seq of random bases and whenever the base |+> & |-> we apply a hadamard gate to invert the 
# qbit in the normal Z-base for |0> & |1> we leave the qbits as they are 
def measure(circuits: Circs, bases: Base) -> None:
    if len(circuits) != len(bases):
        raise Exception("####### Error in the measure function <:> Length of the bases list should be equal to the length of circuits list")

    for circ, base in zip(circuits, bases):
        if base == 1:
            circ.h(0)
        circ.measure(0, 0)



# After both ends have swaped the information we compare them through a normal chanel 
# We count those that don't match (have been intercepted) and consider the percentage from the whole
def filter_qbits(enc_bases: Bases, dec_bases: Bases, bits: list[int], rez: list[int]) -> list[int]:
    indeces = []
    if len(rez) != 0:
        rez.clear()

    for i, bases in enumerate(zip(enc_bases, dec_bases)):
        if bases[0] == bases[1]:
            rez.append(bits[i])
            indeces.append(i)

    return indeces

def print_circuits(qcs:Circs) -> None:
    
    num_circs = len(qcs)
    # Create a grid: 1 row, N columns
    fig, axes = plt.subplots(1, num_circs, figsize=(num_circs * 4, 4))
    
    # If there's only 1 circuit, axes isn't a list, so we wrap it
    if num_circs == 1:
        axes = [axes]
        
    for i, qc in enumerate(qcs):
        qc.draw('mpl', ax=axes[i], scale=0.7)
        axes[i].set_title(f'Circuit {i+1}')
    
    plt.tight_layout()
    plt.show()

#  Implementing it as a sneaky intercept strategy with a random picking rate of 70%
def eavesdropping(qcs: Circs):
    PICKING_RATE = .7
    size = len(qcs)
    eve_choice = np.random.choice([True, False], p=[PICKING_RATE, 1-PICKING_RATE], size=size)

    # measure, write down and resend the qbit
    results: list[tuple[int, int, int]] = []
    for i, qc in enumerate(qcs):
        # deciding eihter to measure or not
        if eve_choice[i]:
            # choose a bases for measurement at random
            base = np.random.randint(0, 2)
            if base == 1:
                qc.h(0)
            qc.measure(0, 0)

            # Simulate the circuit and get the result 
            simulator = AerSimulator()
            trans = qsk.transpile(qc, simulator)
            job = simulator.run(trans, shots=1, memory=True)
            rez = job.result().get_memory(trans)
            results.append((i, rez[0], base))
            # re-prepering the qbit 
            qc_new = QuantumCircuit(1, 1)
            if base == 0:
                if rez[0] == 1:
                    qc_new.x(0)
            else:
                if rez[0] == 1:
                    qc_new.x(0)
                qc_new.h(0)

            # replace the old circuite with the new one 
            qc = qc_new 

    return results


def binary_split(bit_block: Bits, indeces: list[int], a_bits: Bits) -> int:
    if len(indeces) == 1:
        return indeces[0] 
    # recursive break the bit_block into two parts and select the one who presents an odd-error
    break_point = len(indeces) // 2
    # fictitiously sending the indeces of both parts to Alice via a channel and wating for a response
    # here we will compute it response ourselves
    left_parity, right_parity = 0, 0
    for idx in range(break_point):
        left_parity += a_bits[indeces[idx]]

    for idx in range(break_point, len(indeces)):
        right_parity += a_bits[indeces[idx]] 

    # bob's calculations 
    bob_left_parity, bob_right_parity = 0, 0
    for i in range(break_point):
        bob_left_parity += bit_block[i]

    for i in range(break_point, len(indeces)):
        bob_right_parity += bit_block[i]

    # compare the 4 results and decide the block with which to go further in recursion
    if abs(left_parity - bob_left_parity) & 1 == 1:
        return binary_split(bit_block[:break_point], indeces[:break_point], a_bits)
    elif abs(right_parity - bob_right_parity) & 1 == 1:
        return binary_split(bit_block[break_point:], indeces[break_point:], a_bits)
    
def true_parity(a_bits: Bits, indeces: list[int]):
    return sum([b for i,b in enumerate(a_bits) if i in indeces])


def info_reconcill(steps: int, a_bits: Bits, b_bits: Bits, Q:float) -> list[Upper_Block]:

    print(f'Original bits of bob: {b_bits}')
    print(f'Alice bits {a_bits}')
    k = int(.73 * Q)
    indeces = list(range(len(b_bits)))
    blocks: list[Upper_Block] = []

    for _ in range(steps):
        if _ != 0:
            k = k * 2
            # shuffle the b_bits
            np.random.shuffle(indeces)
            print(f'shuffled\n')
        print(f'k: {k}\n')
        # split the bits into block queue them in the list
        copy_k = k
        prev = 0
        while copy_k < len(indeces):
            idx = copy.deepcopy(indeces[prev:copy_k])
            # compute the own parity of the block 
            parity = sum([b for i, b in enumerate(b_bits) if i in idx])
            block = Upper_Block(idx, parity, _)
            blocks.append(block)
            prev = copy_k 
            copy_k += k
            print(f'Pair (prev, copy_k): {prev, copy_k}\n')

        # generate the last block in case the last one is smaller then the others
        if prev < len(indeces):
            idx = copy.deepcopy(indeces[prev:])
            # compute the parity of the block 
            bit_sum = sum([b for i, b in enumerate(b_bits) if i in idx])
            block = Upper_Block(idx, bit_sum, _)
            blocks.append(block)
        
    # compute the parity of the blocks 
    for block in blocks:
        real_parity = true_parity(a_bits, block.indeces)
        block.parity = abs(block.bit_sum - real_parity)

    # try to correct the
    for block in blocks:
        if block.parity % 2 == 1:
            bits = [b for i, b in  enumerate(b_bits) if i in block.indeces]
            idx = binary_split(bits, block.indeces, a_bits)
            print(f'In the block {block.id} | I got the index of the next wrong bit {idx}') 
            # correct the bit on that index and update the blocks parity
            flag = 1 if b_bits[idx] == 0 else -1
            b_bits[idx] = 1 if b_bits[idx] == 0 else 0
            for block in blocks:
                if idx in block.indeces:
                    block.parity += flag
                    block.bit_sum += flag 
                print(f'Updates block: {block.id} | parity: {block.parity}')
    print(f'Reconcille bits of bob: {b_bits}')
    return blocks

def gen_blocks(b_bits: Bits, Q:float) -> list[Upper_Block]:
    indeces = list(range(len(b_bits)))
    start, end = 0, 0
    k = int(Q * .73)
    it = 0
    while k < len(b_bits):
        if it != 0:
            k *= 2 
            np.random.shuffle()


def recon(steps: int, a_bits: Bits, b_bits: Bits, Q:float):
    blocks: list[Upper_Block] = []
    queue = deque()




def privacy_amplification():
    pass

# Implement a function for the whole circuite to display at once 
# make sure to also add in the measurements from Eve
def main() -> None:
    # get the bases in a random way
    N = 1024
    a_bases = np.random.randint(0, 2, size=N)
    a_bits = np.random.randint(0, 2, size=N)
    # Bob's
    b_bases = np.random.randint(0, 2, size=N)
    # init the circuits
    qcs: Circs = []
    encode_persending(qcs, a_bases, a_bits)

    # interception
    eves = eavesdropping(qcs)

    measure(qcs, b_bases)
    # print_circuits(qcs)

    # simulating and getting the outputs form the circuits
    # running the circuites to get the results
    simulator = AerSimulator()
    rez = []
    mems = []
    for qc in qcs:
        transpile_circ = qsk.transpile(qc, simulator)
        job = simulator.run(transpile_circ, shots=1, memory=True)
        result = job.result()
        counts = result.get_counts()
        mem = result.get_memory(transpile_circ)
        rez.append(counts)
        mems.append(int(mem[0]))

    print(f'Alices infos: {a_bits}\nand bases: {a_bases}')
    print(f'Bobs infos: {mems}\nand bases: {b_bases}')
    print(f'Eves information: {eves}')
    # comparring the bases and filtering the bits that does not match 
    after_filtering = []
    idx=filter_qbits(a_bases, b_bases, mems, after_filtering)
    print(f'Length of the qbits before filtering {len(mems)} and after filtering {len(after_filtering)}')
    # compare 33% of the bits over a channel to compute the eavesdropping rate (if more then 60% abort)
    how_many = int(.33 * len(after_filtering))
    which_to_compare = np.random.choice(idx, size=how_many)

    # compare the bits
    unmatch = 0
    for i in which_to_compare:
        if a_bits[i] != mems[i]:
            unmatch += 1
    
    print(f'After filtering correct ones based only on bases: {len(idx)}\nAfter filtering the correct one based also on bits {len(idx) - unmatch}')
    print(f'Error rate: {unmatch/len(idx)*100}%')


def run():
    a = [1, 1, 0, 1, 0, 1, 0, 1, 0, 0]
    b = [1, 1, 1, 1, 0, 1]
    indeces = [4, 5, 6, 7, 8, 9]
    # print(binary_split(b, indeces, a))
    Q = 6.3
    s = 5 
    b = [0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1]
    a = [0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1]
    for _ in info_reconcill(s, a, b, Q):
        print(_)





In [ ]:
# couple of tips to consider when designing the functions 
# CASCADE 
# ---------------------------
# The iteration number i. Early iterations have smaller block sizes (and hence more blocks) than later iterations.
# The estimated quantum bit error rate Q. The higher the quantum bitter error rate, the smaller the block size.
#  ---------------------------
# 
#  formulas for the block size are k1 = 0.73 * Q
#                                  k2 = k1 * 2
#                                  k3 = k2 * 2
#                                  ...........

#  On odd-error Blocks we run the Binary algorithm
#  to find and correct exactly one bit in the key 
# 

In [69]:
# Cascade Algo
from collections import deque 


def get_true_parity(a_bits: Bits, block: Upper_Block) -> int:
    return sum([a_bits[i] for i in block.indeces])


def block_gen(a_bits: Bits, b_bits: Bits, k: int, ite: int, first_ite_flag: bool = True) -> list[Upper_Block]:
    blocks: list[Upper_Block] = []
    indeces = list(range(len(b_bits)))
    step = 0 
    while step + k < len(b_bits):
        if not first_ite_flag:
            # shuffle the indeces 
            np.random.shuffle(indeces)
        # break down the indeces' list
        # and set the iteration it was made in
        bit_sum = sum([b_bits[idx] for idx in indeces[step:step+k]])
        block = Upper_Block(indeces[step:step+k], bit_sum, ite)
        blocks.append(block)
        # set the true parity
        block.parity = abs(get_true_parity(a_bits, block) - bit_sum)

        # proceed with iteration
        step += k 
    
    # last check for a Upper Block of fewer bits
    if step < len(b_bits):
        bit_sum = sum([b_bits[idx] for idx in indeces[step:]])
        block = Upper_Block(indeces[step:], bit_sum, ite)
        blocks.append(block)
        block.parity = abs(get_true_parity(a_bits, block) - bit_sum)
    return blocks




def reconcille(a_bits: Bits, b_bits: Bits, Q: float): 
    k = int(Q * .73) 
    iteration = 1
    q: deque[Upper_Block] = deque()
    all_blocks: list[Upper_Block] = []
    while k < len(b_bits):
        # create the blocks 
        blocks = block_gen(a_bits, b_bits, k, iteration, iteration == 1)
        
        for block in blocks:
            all_blocks.append(block)
            q.append(block)
        # find the blocks that can be reconcille 
        recon_idx = []
        for block in blocks:
            if block.parity % 2 == 1:
                idx = binary_split(b_bits, block.indeces, a_bits)
                print(f'idx: {idx}')
                print(f'________________ Old Block ___________________\n')
                print(block)
                print([a_bits[idx] for idx in block.indeces])
                print([b_bits[idx] for idx in block.indeces])
                print('#######################################')
                recon_idx.append(idx)
                # updeting the current block
                old_bit = b_bits[idx]
                new_bit = 1 if b_bits[idx] == 0 else 0 
                # update Bob's bits
                b_bits[idx] = new_bit
                block_update(a_bits, block, old_bit, new_bit)
                print(f'________________ New Block ___________________\n')
                print(block)
                print('#######################################')
        k *= 2
        iteration += 1
    print(f'________________ The Queue ___________________\n')
    print(q)

def block_update(a_bits: Bits, block: Upper_Block, old_bit: Bit, new_bit: Bit):
    s = new_bit - old_bit
    block.bit_sum += s 
    block.parity = abs(get_true_parity(a_bits, block) - block.bit_sum)

# after the iteration has ended and the possible odd-parity blocks have been reconcilled 
# we have to iterate through all old blocks and check if we can reconcille new bits
def cascading(q: deque[Upper_Block], ite: int):
    block = q.popleft()
    while block.iter < ite and len(q) != 0: 
        if block.parity % 2 == 1:
            pass 
            # reconcille()
        # put the block back, but at the end 
        q.append(block)
        block = q.popleft()

    # put the last block - which may have ended the loop back in the queue 
    q.append(block)


b = [0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1]
a = [0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1]
Q = 6.3
Upper_Block.reset()
reconcille(a, b, Q)

idx: 5
________________ Old Block ___________________

Block: BLK_1 | Indeces: [4, 5, 6, 7] | Iter: 1 | Bit_Sum: 3 | Parity: 1
[0, 0, 1, 1]
[1, 1, 1, 0]
#######################################
________________ New Block ___________________

Block: BLK_1 | Indeces: [4, 5, 6, 7] | Iter: 1 | Bit_Sum: 2 | Parity: 0
#######################################
idx: 9
________________ Old Block ___________________

Block: BLK_2 | Indeces: [8, 9, 10, 11] | Iter: 1 | Bit_Sum: 2 | Parity: 1
[0, 0, 1, 0]
[0, 1, 1, 0]
#######################################
________________ New Block ___________________

Block: BLK_2 | Indeces: [8, 9, 10, 11] | Iter: 1 | Bit_Sum: 1 | Parity: 0
#######################################
idx: 17
________________ Old Block ___________________

Block: BLK_4 | Indeces: [16, 17, 18, 19] | Iter: 1 | Bit_Sum: 4 | Parity: 3
[0, 0, 0, 1]
[1, 1, 1, 1]
#######################################
________________ New Block ___________________

Block: BLK_4 | Indeces: [16, 17, 18, 19] | I

In [ ]:
# teleportation protocol
import qiskit as qsk
from qiskit_aer import AerSimulator
from qiskit import QuantumCircuit

type Bit = 0 | 1

def transfer_into(bit: Bit) -> Bit:
    qc = QuantumCircuit(3, 3)

    if bit == 1:
        qc.x(0)

    qc.barrier()
    # prepare the entangled state
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier()
    # execute the teleportation
    qc.cx(0, 1)
    qc.h(0)
    # measurements
    qc.measure(0, 0)
    qc.measure(1, 1)
    # apply cx and cz according to the measurements
    qc.cx(1, 2)
    qc.cz(0, 2)
    qc.barrier()
    # final measurement 
    qc.measure(2, 2)
    # simulate 
    simulator = AerSimulator()
    transp = qsk.transpile(qc, simulator)
    results = simulator.run(transp, shots=1, memory=True).result()
    counts = results.get_counts()
    print(counts)
    memory = results.get_memory()
    # getting the bit we are interested in
    print(memory[0][:1])

transfer_into(1)


{'100': 1}
1


In [67]:
a = [0, 0, 0, 0, 1, 0, 1, 0]
b =[0, 0, 1, 0, 1, 0, 1, 0]
inx = list(range(len(b)))
binary_split(b, inx, a)

2